# Neural Networks and LLMs:

Today we'll work from Traditional ML to Neural Networks to Large Language Models!!

## Imports and Loading Dataset:

In [1]:
# Imports:
import os

import pylab as p
from dotenv import load_dotenv
from huggingface_hub import login
from pricer.evaluator import evaluate
from litellm import completion
from pricer.items import Item
import numpy as np
from tqdm.notebook import tqdm
from sklearn.feature_extraction.text import HashingVectorizer
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.model_selection import train_test_split

In [2]:
# Logging in to Hugging Face:
load_dotenv(override= True)
hf_token = os.getenv('HF_TOKEN')
login(token= hf_token, add_to_git_credential= True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [3]:
# Loading Dataset (Full Version):
username = 'ed-donner'
dataset = f"{username}/items_full"
train, val, test = Item.from_hub(dataset_name= dataset)
print(f"Loaded {len(train)} training items, {len(val)} validation items, {len(test)} test items")

Loaded 800000 training items, 10000 validation items, 10000 test items


## Simple Deep Neural Network:

In [4]:
# Documents (item Summary) and Prices:
y = np.array([float(item.price) for item in train])
documents = [item.summary for item in train]

In [5]:
# Using the HashingVectorizer for a Bag of Words Model
# Using binary=True with the CountVectorizer makes "one-hot vectors"

np.random.seed(42)
vectorizer = HashingVectorizer(n_features= 5000, stop_words= 'english', binary= True)
X = vectorizer.fit_transform(documents)

### Defining The Neural Network using pytorch:

In [6]:
class NeuralNetwork(nn.Module):

    def __init__(self, input_size):
        super(NeuralNetwork, self).__init__()

        self.layer1 = nn.Linear(input_size, 128)
        self.layer2 = nn.Linear(128, 64)
        self.layer3 = nn.Linear(64, 64)
        self.layer4 = nn.Linear(64, 64)
        self.layer5 = nn.Linear(64, 64)
        self.layer6 = nn.Linear(64, 64)
        self.layer7 = nn.Linear(64, 64)
        self.layer8 = nn.Linear(64, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        output1 = self.relu(self.layer1(x))
        output2 = self.relu(self.layer2(output1))
        output3 = self.relu(self.layer3(output2))
        output4 = self.relu(self.layer4(output3))
        output5 = self.relu(self.layer5(output4))
        output6 = self.relu(self.layer6(output5))
        output7 = self.relu(self.layer7(output6))
        output8 = self.layer8(output7)
        return output8

### Converting Data to Pytorch tensors and Batching:

In [7]:
X_train_tensor = torch.FloatTensor(X.toarray())
y_train_tensor = torch.FloatTensor(y).unsqueeze(1)

# Train Test Split:
X_train, X_val, y_train, y_val = train_test_split(X_train_tensor, y_train_tensor, test_size= 0.1, random_state= 42)

# Data Loader:
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# Initializing The Model:
input_size = X_train_tensor.shape[1]
model = NeuralNetwork(input_size)

In [8]:
# Number of Parameters in the Neural Network:

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Number of trainable parameters: {trainable_params:,}')

Number of trainable parameters: 669,249


### Defining Loss Function and Optimizer:

In [9]:
loss_function = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

### Training Neural Network:

In [10]:
EPOCHS = 2

for epoch in range(EPOCHS):
    model.train()

    for batch_X, batch_y in tqdm(train_loader):
        optimizer.zero_grad()

        # The next 4 lines are the 4 stages of training: forward pass, loss calculation, backward pass, optimize
        outputs = model(batch_X)
        loss = loss_function(outputs, batch_y)
        loss.backward()
        optimizer.step()

    model.eval()

    with torch.no_grad():
        val_outputs = model(X_val)
        val_loss = loss_function(val_outputs, y_val)

    print(f'Epoch [{epoch+1}/{EPOCHS}], Train Loss: {loss.item():.3f}, Val Loss: {val_loss.item():.3f}')

  0%|          | 0/11250 [00:00<?, ?it/s]

Epoch [1/2], Train Loss: 13404.397, Val Loss: 12525.666


  0%|          | 0/11250 [00:00<?, ?it/s]

Epoch [2/2], Train Loss: 20886.377, Val Loss: 11135.732


### Function to Evaluate Neural Network:

In [11]:
def neural_network_eval(item):
    model.eval()
    with torch.no_grad():
        vector = vectorizer.transform([item.summary])
        vector = torch.FloatTensor(vector.toarray())
        result = model(vector)[0].item()
    return max(0, result)

### Evaluating Neural Network:

In [12]:
evaluate(function= neural_network_eval,
         data= test,
         size= 200,
         workers= 5)

  0%|          | 0/200 [00:00<?, ?it/s]

$73 $98 $15 $32 $73 $69 $85 $70 $50 $116 $153 $231 $50 $35 $61 $9 $13 $12 $53 $22 $48 $37 $93 $29 $210 $152 $107 $20 $187 $34 $92 $107 $76 $32 $1 $360 $21 $59 $98 $58 $136 $80 $8 $10 $41 $54 $33 $13 $40 $1 $48 $36 $164 $23 $51 $109 $29 $180 $38 $33 $121 $1 $39 $4 $267 $33 $12 $369 $23 $132 $10 $24 $51 $98 $20 $26 $115 $10 $16 $28 $70 $50 $62 $48 $18 $162 $108 $65 $51 $70 $20 $12 $2 $8 $35 $112 $5 $59 $17 $175 $49 $36 $10 $43 $21 $41 $56 $239 $15 $7 $39 $66 $173 $29 $22 $188 $39 $100 $28 $76 $30 $118 $49 $50 $76 $30 $32 $73 $1 $20 $35 $43 $60 $20 $61 $27 $80 $12 $15 $10 $2 $46 $25 $122 $35 $82 $31 $279 $62 $1 $9 $85 $4 $22 $26 $64 $54 $1 $44 $19 $39 $6 $3 $17 $262 $17 $109 $33 $24 $63 $33 $19 $281 $39 $8 $91 $36 $4 $12 $77 $93 $5 $57 $28 $56 $55 $83 $52 $42 $25 $51 $34 $5 $121 $12 $17 $107 $20 $18 $1 

## Using Frontier LLM to Guess Item Prices:

### Function to Prepare Messages for an item:

In [13]:
def messages_for(item):
    message =  f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"

    return [{'role': 'user', 'content': message}]

In [14]:
messages_for(test[0])

[{'role': 'user',
  'content': 'Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Excess V2 Distortion/Modulation Pedal  \nCategory: Music Pedals  \nBrand: Old Blood Noise  \nDescription: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  \nDetails: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.'}]

### Using OpenAI Frontier Models for Price Guessing:

In [15]:
def gpt_4_1_nano(item):
    response = completion(
        model= 'openai/gpt-4.1-nano',
        messages= messages_for(item),
    )

    return response.choices[0].message.content

In [16]:
# Evaluating GPT-4.1-Nano:
evaluate(function= gpt_4_1_nano,
         data= test,
         size= 200,
         workers= 1)

  0%|          | 0/200 [00:00<?, ?it/s]

$1 $34 $5 $10 $120 $80 $24 $65 $1 $870 $137 $20 $30 $4 $29 $13 $41 $10 $40 $31 $84 $26 $15 $25 $182 $254 $705 $5 $251 $64 $30 $60 $10 $50 $35 $119 $90 $21 $36 $13 $175 $45 $20 $5 $70 $0 $22 $3 $75 $82 $20 $105 $175 $10 $197 $16 $8 $110 $48 $3 $116 $33 $56 $40 $571 $0 $40 $295 $25 $74 $17 $8 $130 $4 $5 $21 $76 $0 $8 $3 $30 $4 $15 $74 $11 $25 $32 $56 $40 $21 $13 $20 $5 $20 $2 $78 $1 $43 $50 $325 $20 $3 $12 $11 $101 $32 $10 $350 $14 $1 $20 $136 $44 $53 $54 $80 $5 $5 $64 $47 $29 $211 $50 $84 $0 $10 $20 $51 $59 $89 $179 $13 $15 $5 $85 $0 $55 $10 $28 $32 $16 $100 $20 $14 $134 $118 $15 $690 $115 $13 $3 $94 $22 $10 $4 $129 $71 $41 $20 $15 $411 $18 $7 $2 $340 $7 $752 $34 $15 $5 $10 $3 $120 $8 $27 $201 $3 $57 $34 $7 $46 $15 $150 $99 $100 $8 $73 $17 $0 $2 $15 $99 $15 $11 $40 $70 $30 $30 $1 $1 